# Week 9: Encoder‑Decoder Generation (T5)

Goal: learn **seq2seq** (encoder‑decoder) generation via the **text‑to‑text** framing popularized by T5.

We’ll do:
- a tiny T5-style encoder-decoder in pure PyTorch
- a small translation task (English → German)
- CPU‑friendly fine‑tuning defaults (small slices + few steps)

Notes:
- First run will download a dataset/model from Hugging Face.
- Keep dataset sizes and steps small for CPU.

In [ ]:
# Setup
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

seed = 204
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

## Part A — What changes vs GPT?

**Decoder‑only (GPT):** one stack, causal self‑attention, predicts next token.

**Encoder‑decoder (T5):**
- **Encoder** reads the full source sequence with bidirectional self‑attention.
- **Decoder** generates the target with causal self‑attention **and** cross‑attention into the encoder states.
- Training uses teacher forcing: decoder sees the gold prefix, learns to predict the next target token.

## Part B — Tiny encoder-decoder Transformer in pure PyTorch

This mirrors the GPT scratch build, but now we make the seq2seq mechanics concrete:
- an **encoder** with bidirectional self-attention over the source tokens
- a **decoder** with causal self-attention over the generated target prefix
- **cross-attention** from decoder states into encoder states
- teacher forcing with a shifted-right decoder input
- token-level generation loss over the target sequence

This is T5-style architecture intuition, not a faithful reproduction of every T5 detail.

In [ ]:
@dataclass
class TinyT5Config:
    vocab_size: int
    max_source_len: int = 16
    max_target_len: int = 16
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.1
    pad_id: int = 0
    bos_id: int = 1


class MultiHeadAttention(nn.Module):
    def __init__(self, cfg: TinyT5Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.q = nn.Linear(cfg.d_model, cfg.d_model)
        self.k = nn.Linear(cfg.d_model, cfg.d_model)
        self.v = nn.Linear(cfg.d_model, cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def split_heads(self, x):
        bsz, seq_len, d_model = x.shape
        x = x.view(bsz, seq_len, self.n_heads, self.head_dim)
        return x.transpose(1, 2)

    def forward(self, query, key_value, mask=None):
        q = self.split_heads(self.q(query))
        k = self.split_heads(self.k(key_value))
        v = self.split_heads(self.v(key_value))

        scores = q @ k.transpose(-2, -1) / (self.head_dim ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(query.size(0), query.size(1), -1)
        return self.proj(out)


class FeedForward(nn.Module):
    def __init__(self, cfg: TinyT5Config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model, 4 * cfg.d_model),
            nn.GELU(),
            nn.Linear(4 * cfg.d_model, cfg.d_model),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg: TinyT5Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = MultiHeadAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = FeedForward(cfg)

    def forward(self, x, src_mask):
        x = x + self.self_attn(self.ln1(x), self.ln1(x), src_mask)
        x = x + self.ff(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, cfg: TinyT5Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = MultiHeadAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = MultiHeadAttention(cfg)
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ff = FeedForward(cfg)

    def forward(self, x, enc, tgt_mask, src_mask):
        x = x + self.self_attn(self.ln1(x), self.ln1(x), tgt_mask)
        x = x + self.cross_attn(self.ln2(x), enc, src_mask)
        x = x + self.ff(self.ln3(x))
        return x


class TinyT5(nn.Module):
    def __init__(self, cfg: TinyT5Config):
        super().__init__()
        self.cfg = cfg
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.src_pos_emb = nn.Embedding(cfg.max_source_len, cfg.d_model)
        self.tgt_pos_emb = nn.Embedding(cfg.max_target_len, cfg.d_model)
        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size)

    def make_src_mask(self, src_ids):
        return (src_ids != self.cfg.pad_id).view(src_ids.size(0), 1, 1, src_ids.size(1))

    def make_tgt_mask(self, tgt_ids):
        bsz, tgt_len = tgt_ids.shape
        pad_mask = (tgt_ids != self.cfg.pad_id).view(bsz, 1, 1, tgt_len)
        causal = torch.tril(torch.ones(tgt_len, tgt_len, device=tgt_ids.device, dtype=torch.bool))
        return pad_mask & causal.view(1, 1, tgt_len, tgt_len)

    def shift_right(self, labels):
        bos = torch.full((labels.size(0), 1), self.cfg.bos_id, device=labels.device, dtype=labels.dtype)
        return torch.cat([bos, labels[:, :-1]], dim=1)

    def encode(self, src_ids):
        bsz, src_len = src_ids.shape
        pos = torch.arange(src_len, device=src_ids.device).unsqueeze(0)
        x = self.token_emb(src_ids) + self.src_pos_emb(pos)
        src_mask = self.make_src_mask(src_ids)
        for block in self.encoder:
            x = block(x, src_mask)
        return x, src_mask

    def decode(self, decoder_ids, enc, src_mask):
        bsz, tgt_len = decoder_ids.shape
        pos = torch.arange(tgt_len, device=decoder_ids.device).unsqueeze(0)
        x = self.token_emb(decoder_ids) + self.tgt_pos_emb(pos)
        tgt_mask = self.make_tgt_mask(decoder_ids)
        for block in self.decoder:
            x = block(x, enc, tgt_mask, src_mask)
        return self.lm_head(self.ln(x))

    def forward(self, src_ids, labels=None):
        enc, src_mask = self.encode(src_ids)
        if labels is None:
            raise ValueError("Pass target labels during training so the decoder can be shifted right.")
        decoder_ids = self.shift_right(labels)
        logits = self.decode(decoder_ids, enc, src_mask)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=self.cfg.pad_id,
        )
        return logits, loss


In [ ]:
# Tiny synthetic seq2seq task: map source tokens to the reversed target sequence.
# This keeps the objective local and fast while exercising encoder, decoder, and cross-attention.
PAD, BOS, EOS = 0, 1, 2
vocab_size = 32
src_len = 8
tgt_len = src_len + 1  # reversed source plus EOS

def make_reverse_batch(batch_size=64):
    src = torch.randint(3, vocab_size, (batch_size, src_len), device=device)
    labels = torch.cat(
        [torch.flip(src, dims=[1]), torch.full((batch_size, 1), EOS, device=device)],
        dim=1,
    )
    return src, labels

cfg = TinyT5Config(
    vocab_size=vocab_size,
    max_source_len=src_len,
    max_target_len=tgt_len,
    d_model=128,
    n_heads=4,
    n_layers=2,
    dropout=0.1,
    pad_id=PAD,
    bos_id=BOS,
)
tiny_t5 = TinyT5(cfg).to(device)
optimizer = torch.optim.AdamW(tiny_t5.parameters(), lr=3e-4)

tiny_t5.train()
for step in range(80):
    src, labels = make_reverse_batch(batch_size=64)
    logits, loss = tiny_t5(src, labels)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(f"step {step:02d} | loss {loss.item():.3f}")

tiny_t5.eval()
src, labels = make_reverse_batch(batch_size=1)
with torch.no_grad():
    logits, loss = tiny_t5(src, labels)
    pred = logits.argmax(dim=-1)

print("source: ", src[0].tolist())
print("target: ", labels[0].tolist())
print("pred:   ", pred[0].tolist())


## Part C — Hugging Face T5 translation (CPU‑friendly)

We use the dataset from the course roadmap: `opus100` (en‑de).

To keep this realistic on CPU we:
- take small slices
- cap max source/target length
- train for a small number of steps

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

hf_model_name = "t5-small"

# CPU-friendly sizes
n_train = 4000
n_val = 500

max_source_len = 128
max_target_len = 128

raw = load_dataset("opus100", "en-de")
raw

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(hf_model_name)

prefix = "translate English to German: "

def preprocess(batch):
    src_texts = [prefix + ex["en"] for ex in batch["translation"]]
    tgt_texts = [ex["de"] for ex in batch["translation"]]

    model_inputs = tokenizer(
        src_texts,
        max_length=max_source_len,
        truncation=True,
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            tgt_texts,
            max_length=max_target_len,
            truncation=True,
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tok = raw.map(preprocess, batched=True, remove_columns=raw["train"].column_names)

train_ds = tok["train"].shuffle(seed=seed).select(range(min(n_train, len(tok["train"])) ))
val_ds = tok["validation"].shuffle(seed=seed).select(range(min(n_val, len(tok["validation"])) ))
train_ds, val_ds

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(hf_model_name).to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

out_dir = "./models/week9_t5small_opus100_en_de"
os.makedirs(out_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=out_dir,
    overwrite_output_dir=True,
    max_steps=300,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=30,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=seed,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer

In [ ]:
# CPU-realistic run: keep this small
# trainer.train()

# If you already trained, point to a checkpoint folder here:
# ckpt = "./models/week9_t5small_opus100_en_de/checkpoint-300"
# model = AutoModelForSeq2SeqLM.from_pretrained(ckpt).to(device)

print("Ready: uncomment trainer.train() to fine-tune.")

In [ ]:
# Translation demo (works before/after fine-tuning)
from transformers import set_seed

set_seed(seed)
src = "I really enjoyed this movie, but the ending was disappointing."
inp = tokenizer(prefix + src, return_tensors="pt").to(device)

gen = model.generate(
    **inp,
    max_new_tokens=80,
    num_beams=4,
)

print("EN:", src)
print("DE:", tokenizer.decode(gen[0], skip_special_tokens=True))